In [ ]:
#######################################################
# SHARE THE FOLLOWING CODE WITH CONTESTANTS
#######################################################

import os
from io import BytesIO

import numpy as np
import pandas as pd
from PIL import Image

from huggingface_hub import login, hf_hub_download, list_repo_files

login(token)

repo_id = "usaaio-official/2026_USAAIO_Round2_sample"
dataset_subdir = "Shapes"

def load_image_as_numpy(image_path):
    img = Image.open(image_path).convert("RGB")
    return np.array(img, dtype=np.uint8)


def download_and_load_image_folder(repo_id, \
                                   folder_path, repo_type="dataset"):
    all_files = list_repo_files(repo_id=repo_id, \
                                repo_type=repo_type)

    png_files = sorted(
        [
            f for f in all_files
            if f.startswith(folder_path + "/") \
            and f.lower().endswith(".png")
        ]
    )

    images = []
    image_names = []

    for file_path in png_files:
        local_path = hf_hub_download(
            repo_id=repo_id,
            filename=file_path,
            repo_type=repo_type,
        )
        images.append(load_image_as_numpy(local_path))
        image_names.append(os.path.basename(file_path))

    images = np.stack(images, axis=0)
    return images, image_names


def load_labels_csv(repo_id, csv_path, repo_type="dataset"):
    local_path = hf_hub_download(
        repo_id=repo_id,
        filename=csv_path,
        repo_type=repo_type,
    )
    return pd.read_csv(local_path)

train_images, train_image_names = download_and_load_image_folder(
    repo_id=repo_id,
    folder_path=f"{dataset_subdir}/train_images",
    repo_type="dataset",
)

val_images, val_image_names = download_and_load_image_folder(
    repo_id=repo_id,
    folder_path=f"{dataset_subdir}/val_images",
    repo_type="dataset",
)

test_images, test_image_names = download_and_load_image_folder(
    repo_id=repo_id,
    folder_path=f"{dataset_subdir}/test_images",
    repo_type="dataset",
)

train_labels = load_labels_csv(
    repo_id=repo_id,
    csv_path=f"{dataset_subdir}/train_labels.csv",
    repo_type="dataset",
)

val_labels = load_labels_csv(
    repo_id=repo_id,
    csv_path=f"{dataset_subdir}/val_labels.csv",
    repo_type="dataset",
)

sample_submission_path = hf_hub_download(
    repo_id=repo_id,
    filename=f"{dataset_subdir}/sample_submission.csv",
    repo_type="dataset",
)
sample_submission = pd.read_csv(sample_submission_path)

train_label_map = dict(zip(train_labels["image_name"], \
                           train_labels["label"]))
val_label_map = dict(zip(val_labels["image_name"], \
                         val_labels["label"]))

y_train = np.array([train_label_map[name] \
                    for name in train_image_names], dtype=np.int64)
y_val = np.array([val_label_map[name] \
                  for name in val_image_names], dtype=np.int64)

X_train = train_images
X_val = val_images
X_test = test_images

In [ ]:
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("X_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)

print("X_test shape:", X_test.shape)
print()
print("Sample submission:")
print(sample_submission.head())

X_train shape: (500, 112, 112, 3)
y_train shape: (500,)
X_val shape: (100, 112, 112, 3)
y_val shape: (100,)
X_test shape: (200, 112, 112, 3)

Sample submission:
   image_name  label
0  test_0.png    NaN
1  test_1.png    NaN
2  test_2.png    NaN
3  test_3.png    NaN
4  test_4.png    NaN


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from sklearn.metrics import f1_score

y_train = y_train.astype(np.int64)
y_val = y_val.astype(np.int64)
class easyDataset(Dataset):
  def __init__(self, images, labels = None):
    self.images = images
    self.labels = labels
    self.transform = transforms.ToTensor()

  def __len__(self):
    return len(self.images)

  def __getitem__(self, idx):
    image = self.images[idx]
    image = self.transform(image)

    if self.labels is not None:
      return image, self.labels[idx]
    return image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet18(weights='DEFAULT')
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

train_loader = DataLoader(easyDataset(X_train, y_train), batch_size=32, shuffle=True)
val_loader = DataLoader(easyDataset(X_val, y_val), batch_size=32)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

def train_model(epochs=10):
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0

        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

            predicted = torch.argmax(outputs, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = model(imgs)
                loss = criterion(outputs, labels)

                val_loss += loss.item()

                predicted = torch.argmax(outputs, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        train_acc = 100 * train_correct / train_total
        val_acc = 100 * val_correct / val_total

        print(f"Epoch {epoch+1}/{epochs}")
        print(f"Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.2f}%")
        print(f"Val Loss:   {avg_val_loss:.4f} | Val Acc:   {val_acc:.2f}%")
        print("-" * 30)

train_model(epochs=10)

Epoch 1/10
Train Loss: 0.1358 | Train Acc: 93.80%
Val Loss:   0.0030 | Val Acc:   100.00%
------------------------------
Epoch 2/10
Train Loss: 0.0075 | Train Acc: 99.80%
Val Loss:   0.0005 | Val Acc:   100.00%
------------------------------
Epoch 3/10
Train Loss: 0.0032 | Train Acc: 100.00%
Val Loss:   0.0004 | Val Acc:   100.00%
------------------------------
Epoch 4/10
Train Loss: 0.0005 | Train Acc: 100.00%
Val Loss:   0.0003 | Val Acc:   100.00%
------------------------------
Epoch 5/10
Train Loss: 0.0006 | Train Acc: 100.00%
Val Loss:   0.0001 | Val Acc:   100.00%
------------------------------
Epoch 6/10
Train Loss: 0.0005 | Train Acc: 100.00%
Val Loss:   0.0001 | Val Acc:   100.00%
------------------------------
Epoch 7/10
Train Loss: 0.0002 | Train Acc: 100.00%
Val Loss:   0.0001 | Val Acc:   100.00%
------------------------------
Epoch 8/10
Train Loss: 0.0002 | Train Acc: 100.00%
Val Loss:   0.0001 | Val Acc:   100.00%
------------------------------
Epoch 9/10
Train Loss: 0.0

In [ ]:
model.eval()
test_predictions = []
from google.colab import files

test_loader = DataLoader(easyDataset(X_test), batch_size=32)

with torch.no_grad():
    for imgs in test_loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        predicted = torch.argmax(outputs, 1)
        test_predictions.extend(predicted.cpu().numpy())

last_name = "YourLastName"
first_name = "YourFirstName"
school = "YourSchoolName"
file_name = f"Shape_Classification_{last_name}_{first_name}_{school}.csv"

df = pd.DataFrame({
    'Id': range(len(test_predictions)),
    'Label': test_predictions
})

df.to_csv(file_name, index=False)

print(f"Submission file saved as: {file_name}")
files.download(file_name)

Submission file saved as: Shape_Classification_YourLastName_YourFirstName_YourSchoolName.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>